In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dog_dataset (1).zip'
unzip_dir = '/content/dog_dataset_new'

os.makedirs(unzip_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

print(f"'{zip_path}' unzipped to '{unzip_dir}'")

'/content/drive/MyDrive/dog_dataset (1).zip' unzipped to '/content/dog_dataset_new'


In [3]:
# Install YOLOv8
!pip install ultralytics
import os
from ultralytics import YOLO
from PIL import Image
from tqdm import tqdm
import shutil

# Load pre-trained YOLOv8 nano (fast & good for dogs)
model = YOLO('yolov8n.pt')  # Auto downloads if not present

# Your dataset path
base_dir = '/content/dog_dataset_new'  # Updated path to the new unzipped dataset

# New cropped dataset path
cropped_base = '/content/cropped_bcs_dataset'
if os.path.exists(cropped_base):
    shutil.rmtree(cropped_base)  # Fresh start
os.makedirs(cropped_base, exist_ok=True)

# Walk through all subfolders and images
total_images = 0
cropped_count = 0

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(root, file)
            total_images += 1

            # YOLO detection
            results = model(img_path, classes=[16])  # class 16 = 'dog' in COCO
            if len(results[0].boxes) > 0:
                # Take the first (largest) dog box
                box = results[0].boxes[0].xyxy[0].cpu().numpy()
                xmin, ymin, xmax, ymax = map(int, box)

                # Open and crop
                with Image.open(img_path) as im:
                    cropped = im.crop((xmin, ymin, xmax, ymax))
                    if cropped.mode != 'RGB':
                        cropped = cropped.convert('RGB')

                    # Replicate folder structure in cropped
                    rel_path = os.path.relpath(root, base_dir)
                    new_dir = os.path.join(cropped_base, rel_path)
                    os.makedirs(new_dir, exist_ok=True)
                    new_path = os.path.join(new_dir, file)
                    cropped.save(new_path)
                    cropped_count += 1

print(f"\nProcessed {total_images} images")
print(f"Successfully cropped {cropped_count} dogs")
print(f"Cropped dataset saved to: {cropped_base}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

image 1/1 /content/dog_dataset_new/dog_dataset1/puppy/overweight/Pomerian/download (2).jpg: 640x480 (no detections), 75.7ms
Speed: 12.0ms preprocess, 75.7ms inference, 65.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /content/dog_dataset_new/dog_dataset1/puppy/overweight/Pomerian/download (4).jpg: 640x416 1 dog, 44.7ms
Speed: 2.9ms preprocess, 44.7ms inference, 35.5ms postprocess per image at shape (1, 3, 640, 416)

image 1/1 /content/dog_dataset_new/dog_dataset1/puppy/overweight/Pomerian/download (3).jpg: 480x640 (no detections), 45.5ms
Speed: 3.4ms preprocess, 45.5ms inference, 1.0ms p

In [8]:
# ==================================================
# FIXED BCS TRAINING FOR NESTED CROPPED DATASET
# ==================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

# -------------------------------------------------
# 1. Path & Nested Loading
# -------------------------------------------------
base_dir = '/content/cropped_bcs_dataset/dog_dataset1'  # Your cropped path

bcs_map = {'underweight': 0, 'healthy': 1, 'overweight': 2}
age_groups = ['puppy', 'adult', 'senior']  # Your age folders

df_list = []

for age in age_groups:
    age_dir = os.path.join(base_dir, age)
    if not os.path.exists(age_dir):
        print(f"Age folder not found: {age_dir}")
        continue

    for bcs_folder in os.listdir(age_dir):
        bcs_dir = os.path.join(age_dir, bcs_folder)
        if not os.path.isdir(bcs_dir) or bcs_folder not in bcs_map:
            continue

        label = bcs_map[bcs_folder]
        print(f"Loading {bcs_folder} ({label}) from {age}...")

        for breed_folder in os.listdir(bcs_dir):
            breed_dir = os.path.join(bcs_dir, breed_folder)
            if not os.path.isdir(breed_dir):
                continue

            for img_file in os.listdir(breed_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(breed_dir, img_file)
                    df_list.append({
                        'image_path': img_path,
                        'bcs_label': label,
                        'age': age,
                        'breed': breed_folder
                    })

df = pd.DataFrame(df_list)
print(f"\nTotal images loaded: {len(df)}")
if len(df) == 0:
    print("No images found — check if cropping succeeded and paths are correct.")
else:
    print("BCS distribution:")
    print(df['bcs_label'].value_counts().rename({0: 'Underweight', 1: 'healthy', 2: 'Overweight'}))

# -------------------------------------------------
# 2. Dataset Class
# -------------------------------------------------
class BCSDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['bcs_label'], dtype=torch.long)
        return img, label

# -------------------------------------------------
# 3. Transforms
# -------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5)
])

test_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['bcs_label'] if len(df['bcs_label'].unique()) > 1 else None, random_state=42)

train_dataset = BCSDataset(train_df, transform=train_transform)
test_dataset = BCSDataset(test_df, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# -------------------------------------------------
# 4. Class Weights (dynamic for existing classes)
# -------------------------------------------------
train_labels = train_df['bcs_label'].values
unique_classes = np.unique(train_labels)
class_weights = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Detected classes: {unique_classes}")
print(f"Class weights: {class_weights}")

# -------------------------------------------------
# 5. Model (dynamic output classes)
# -------------------------------------------------
num_classes = len(unique_classes)
model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(DEVICE)

# -------------------------------------------------
# 6. Loss & Optimizer
# -------------------------------------------------
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# -------------------------------------------------
# 7. Training Loop
# -------------------------------------------------
EPOCHS = 30
best_acc = 0.0
patience = 10
counter = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total if total > 0 else 0
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), '/content/bcs_cropped_model.pth')
        print("   >>> New best model saved!")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

print(f"\nTraining complete! Best val accuracy: {best_acc:.2f}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Loading overweight (2) from puppy...
Loading healthy (1) from puppy...
Loading underweight (0) from puppy...
Loading overweight (2) from adult...
Loading healthy (1) from adult...
Loading underweight (0) from adult...
Loading overweight (2) from senior...
Loading healthy (1) from senior...
Loading underweight (0) from senior...

Total images loaded: 886
BCS distribution:
bcs_label
healthy        313
Underweight    313
Overweight     260
Name: count, dtype: int64
Detected classes: [0 1 2]
Class weights: tensor([0.9440, 0.9440, 1.1346], device='cuda:0')


Epoch 1/30: 100%|██████████| 23/23 [00:13<00:00,  1.70it/s]


Epoch 1/30 - Loss: 1.0855 | Val Acc: 55.62%
   >>> New best model saved!


Epoch 2/30: 100%|██████████| 23/23 [00:13<00:00,  1.70it/s]


Epoch 2/30 - Loss: 1.0243 | Val Acc: 62.36%
   >>> New best model saved!


...

Epoch 18/30 - Loss: 0.4193 | Val Acc: 69.10%


Epoch 19/30: 100%|██████████| 23/23 [00:15<00:00,  1.52it/s]


Epoch 19/30 - Loss: 0.4227 | Val Acc: 68.54%


Epoch 20/30: 100%|██████████| 23/23 [00:15<00:00,  1.51it/s]


Epoch 20/30 - Loss: 0.4200 | Val Acc: 67.42%
Early stopping

Training complete! Best val accuracy: 71.35%


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.transforms import RandAugment
from PIL import ImageFile

# Allow loading truncated/corrupted images
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

# -------------------------------------------------
# 1. Load Data (nested structure)
# -------------------------------------------------
base_dir = '/content/dog_dataset_new/dog_dataset1'

bcs_map = {
    'underweight': 0,
    'healthy': 1,
    'overweight': 2
}

df_list = []

for age in ['puppy', 'adult', 'senior']:
    age_dir = os.path.join(base_dir, age)
    if not os.path.exists(age_dir):
        continue

    for bcs_folder in os.listdir(age_dir):
        if bcs_folder in bcs_map:
            label = bcs_map[bcs_folder]
            bcs_dir = os.path.join(age_dir, bcs_folder)
            for breed in os.listdir(bcs_dir):
                breed_dir = os.path.join(bcs_dir, breed)
                if not os.path.isdir(breed_dir):
                    continue
                for file in os.listdir(breed_dir):
                    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(breed_dir, file)
                        df_list.append({
                            'image_path': img_path,
                            'bcs_label': label
                        })

df = pd.DataFrame(df_list)
print(f"Total images: {len(df)}")
print("BCS distribution:")
print(df['bcs_label'].value_counts().sort_index().rename({
    0: 'Underweight', 1: 'Healthy', 2: 'Overweight'
}))

# -------------------------------------------------
# 2. Robust Dataset (handles bad images)
# -------------------------------------------------
class SafeBCSDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except:
            img = Image.new('RGB', (300, 300), (128, 128, 128))  # fallback

        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['bcs_label'], dtype=torch.long)
        return img, label

# -------------------------------------------------
# 3. Augmentation
# -------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.7, 1.0)),
    RandAugment(num_ops=3, magnitude=12),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.5, 0.5, 0.5, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.7)
])

test_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['bcs_label'], random_state=42)

train_dataset = SafeBCSDataset(train_df, train_transform)
test_dataset = SafeBCSDataset(test_df, test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# -------------------------------------------------
# 4. Class Weights (FIXED: classes as np.array)
# -------------------------------------------------
train_labels = train_df['bcs_label'].values
class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2]),  # ← Fixed: np.array
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Class weights: Underweight={class_weights[0]:.2f}, Healthy={class_weights[1]:.2f}, Overweight={class_weights[2]:.2f}")

# -------------------------------------------------
# 5. Model
# -------------------------------------------------
model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# -------------------------------------------------
# 6. Training
# -------------------------------------------------
EPOCHS = 30
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/bcs_model_final.pth')
        print("   >>> New best model saved!")

print(f"\nFINAL BEST ACCURACY: {best_acc:.2f}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Total images: 1246
BCS distribution:
bcs_label
Underweight    426
Healthy        422
Overweight     398
Name: count, dtype: int64
Class weights: Underweight=0.97, Healthy=0.99, Overweight=1.04


Epoch 1/30: 100%|██████████| 32/32 [00:25<00:00,  1.26it/s]


Epoch 1 - Loss: 1.0723 | Val Acc: 51.20%
   >>> New best model saved!


Epoch 2/30: 100%|██████████| 32/32 [00:20<00:00,  1.55it/s]


Epoch 2 - Loss: 1.0216 | Val Acc: 64.00%
   >>> New best model saved!


...

Epoch 28 - Loss: 0.5693 | Val Acc: 73.60%


Epoch 29/30: 100%|██████████| 32/32 [00:21<00:00,  1.46it/s]


Epoch 29 - Loss: 0.5379 | Val Acc: 74.00%


Epoch 30/30: 100%|██████████| 32/32 [00:21<00:00,  1.51it/s]


Epoch 30 - Loss: 0.5728 | Val Acc: 74.40%

FINAL BEST ACCURACY: 77.20%


In [13]:
# -------------------------------------------------
# 8. Test the BCS Model
# -------------------------------------------------
model.load_state_dict(torch.load('/content/bcs_model_final.pth'))
# model.load_state_dict(torch.load('/content/bcs_cropped_model.pth'))
model.eval()

def predict_bcs(image_path):
    img = test_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
        pred_label = np.argmax(probs)
        conf = probs.max()
    pred = ['Underweight', 'Normal', 'Overweight'][pred_label]
    print(f"Predicted BCS: {pred} (conf: {conf:.2f})")
    print(f"Probs: Under={probs[0]:.2f}, Normal={probs[1]:.2f}, Over={probs[2]:.2f}")

# Test example
predict_bcs('/content/1_Sunder.webp')  # Replace with an image path
predict_bcs('/content/0_Harvey-when-removed-from-the-property.webp')  # Replace with an image path
predict_bcs('/content/overweight-lab-1024x640.jpg')
predict_bcs('/content/labrador-retriever-yellow-sitting.jpg')

Predicted BCS: Overweight (conf: 0.85)
Probs: Under=0.07, Normal=0.08, Over=0.85
Predicted BCS: Underweight (conf: 0.64)
Probs: Under=0.64, Normal=0.17, Over=0.19
Predicted BCS: Overweight (conf: 0.81)
Probs: Under=0.03, Normal=0.16, Over=0.81
Predicted BCS: Normal (conf: 0.68)
Probs: Under=0.25, Normal=0.68, Over=0.07


In [ ]:
Predicted BCS: Overweight (conf: 0.85)
Probs: Under=0.07, Normal=0.08, Over=0.85
Predicted BCS: Underweight (conf: 0.64)
Probs: Under=0.64, Normal=0.17, Over=0.19
Predicted BCS: Overweight (conf: 0.81)
Probs: Under=0.03, Normal=0.16, Over=0.81
Predicted BCS: Normal (conf: 0.68)
Probs: Under=0.25, Normal=0.68, Over=0.07

In [ ]:
Predicted BCS: Overweight (conf: 0.90)
Probs: Under=0.05, Normal=0.05, Over=0.90
Predicted BCS: Underweight (conf: 0.92)
Probs: Under=0.92, Normal=0.06, Over=0.02
Predicted BCS: Overweight (conf: 0.92)
Probs: Under=0.01, Normal=0.07, Over=0.92
Predicted BCS: Normal (conf: 0.74)
Probs: Under=0.21, Normal=0.74, Over=0.05